In [ ]:
!git clone https://github.com/matiimonti/Transformer_project
%cd Transformer_project

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git pull

## Chess Transformer

In [ ]:
import urllib.request
import os
import io
import sys
import importlib
sys.path.insert(0, 'src')
!pip install torch -q
import torch
from torch.utils.data import DataLoader

import pgn_data
importlib.reload(pgn_data)

# Decompressor for the .zip Lichess file
# !pip install zstandard -q
import zstandard as zstd

os.makedirs('data', exist_ok = True)

In [ ]:
# Import Lichess games 
YEAR = '2013'
MONTH = '01'
url = f'https://database.lichess.org/standard/lichess_db_standard_rated_{YEAR}-{MONTH}.pgn.zst'
zst_path = f'data/lichess_{YEAR}_{MONTH}.pgn.zst'
pgn_path = 'data/games.pgn'

print(f'Downloading: {url}')

### Stream download
def download(url, dest):
    def reporthook(count, block_size, total_size):
        mb_done = count * block_size / 1_000_000
        mb_total = total_size / 1_000_000 if total_size > 0 else '?'
        print(f'\r {mb_done:.1f} MB / {mb_total} MB', end='', flush=True)
    urllib.request.urlretrieve(url, dest, reporthook=reporthook)
    print()

download(url, zst_path)
print(f"Compressed file saved: {zst_path}")


In [ ]:
### Decompress file
dctx = zstd.ZstdDecompressor()
with open(zst_path, 'rb') as compressed:
    with open(pgn_path, 'wb') as out_file:
        dctx.copy_stream(compressed, out_file)
print(f'File saved to: {pgn_path}')
print(f'File size: {os.path.getsize(pgn_path) / 1_000_000:.1f} MB')

In [ ]:
## Example check
with open(pgn_path) as f:
    sample = "".join([next(f) for _ in range(30)])

print("\nFirst 30 lines:")
print(sample)

### Parse Data 

In [ ]:
from pgn_data import parse_pgn, ChessTokenizer

SAMPLE_BYTES = 500_000

with open(pgn_path, 'r', encoding='utf-8', errors='ignore') as f:
    sample_text = f.read(SAMPLE_BYTES)

# Trim to last complete game (avoid cutting mid-game)
last_newline = sample_text.rfind('\n\n')
if last_newline != -1:
    sample_text = sample_text[:last_newline]

sample_games = parse_pgn(sample_text)
print(f'Games parsed from sample: {len(sample_games)}')
print(f'Avg moves per game:       {sum(len(g) for g in sample_games) / len(sample_games):.1f}')


In [ ]:
# Inspect the first 3 games
for i, game in enumerate(sample_games[:3]):
    print(f'--- Game {i+1} ({len(game)} moves) ---')
    print(' '.join(game))
    print()

In [ ]:
# Tokenizer
tok = ChessTokenizer()
tok.build_from_games(sample_games)

print(f"Vocab size: {tok.vocab_size}")

# Check on first game
encoded = tok.encode(sample_games[0])
decoded = tok.decode(encoded)

print(f"Original: {sample_games[0][:5]}")
print(f"Encoded:  {encoded[:7]}")
print(f"Decoded:  {decoded[:7]}")

# The key check — decoded middle should match original exactly
assert decoded[1:-1] == sample_games[0], "Round-trip failed!"
print("Round-trip passed.")

In [ ]:
from pgn_data import ChessDataset

encoded_games = [tok.encode(g) for g in sample_games]
ds = ChessDataset(encoded_games, seq_len=128, pad_id=tok.pad_id)

inp, tgt = ds[0]

print(f"Dataset size: {len(ds)} samples")
print(f"Input shape:  {inp.shape}")
print(f"Target shape: {tgt.shape}")
print(f"Input:  {inp[:10].tolist()}")
print(f"Target: {tgt[:10].tolist()}")

In [ ]:
from pgn_data import load_data

# load_data now splits at the game level and returns two datasets
train_dataset, val_dataset, tokenizer = load_data(
    pgn_path=pgn_path,
    seq_len=128,
)

# Verify the DataLoader works
loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2)
x, y = next(iter(loader))

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples:   {len(val_dataset)}")
print(f"Batch shape:   {x.shape}")
print(f"Vocab size:    {tokenizer.vocab_size}")

## Core Attention

In [ ]:
from attention import scaled_dot_product_attention

B, heads, T, head_dim = 2, 4, 16, 32

q = torch.randn(B, heads, T, head_dim)
k = torch.randn(B, heads, T, head_dim)
v = torch.randn(B, heads, T, head_dim)

out = scaled_dot_product_attention(q, k, v)

print(f"Input shape:  {q.shape}")
print(f"Output shape: {out.shape}")
assert out.shape == q.shape
print("scaled_dot_product_attention ✓")

In [ ]:
from attention import causal_mask

mask = causal_mask(6, device=torch.device('cpu'))
print(mask.int())

In [ ]:
from attention import MultiHeadAttention

mha  = MultiHeadAttention(d_model=128, n_heads=4)
x = torch.randn(2, 16, 128, requires_grad=True)
mask = causal_mask(16, device=torch.device('cpu'))

out  = mha(x, mask=mask)

print(f"Input shape:  {x.shape}")
print(f"Output shape: {out.shape}")
assert out.shape == x.shape, "Shape mismatch"

loss = out.sum()
loss.backward()
assert x.grad is not None, "No gradients flowing"
print(f"Gradient shape: {x.grad.shape}")
print("MultiHeadAttention ✓")

In [ ]:
from attention import MultiHeadAttention, causal_mask
from model import FeedForward, TransformerBlock

# Test FeedForward
ff  = FeedForward(d_model=128)
x   = torch.randn(2, 16, 128)
out = ff(x)
assert out.shape == x.shape
print(f"FeedForward output: {out.shape} ✓")

# Test TransformerBlock
attn  = MultiHeadAttention(d_model=128, n_heads=4)
block = TransformerBlock(d_model=128, attention=attn)
x     = torch.randn(2, 16, 128, requires_grad=True)
mask  = causal_mask(16, device=torch.device('cpu'))
out   = block(x, mask=mask)

assert out.shape == x.shape
loss  = out.sum()
loss.backward()
assert x.grad is not None
print(f"TransformerBlock output: {out.shape} ✓")
print("Gradients flow")

In [ ]:
import attention; importlib.reload(attention)
import model;     importlib.reload(model)

from attention import RoPEMultiHeadAttention, causal_mask
from model import ChessTransformer

# --- 1. Shape + gradient check ---
rope_attn = RoPEMultiHeadAttention(d_model=128, n_heads=4, max_seq_len=256)
x    = torch.randn(2, 16, 128, requires_grad=True)
mask = causal_mask(16, device=torch.device('cpu'))
out  = rope_attn(x, mask=mask)

assert out.shape == x.shape, f"Shape mismatch: {out.shape}"
out.sum().backward()
assert x.grad is not None
print(f"RoPEMultiHeadAttention output: {out.shape} ✓")
print("Gradients flow ✓")

# --- 2. Sanity-check: cos/sin cache shape (max_seq_len, head_dim) ---
assert rope_attn.rope_cos.shape == (256, 32), f"Unexpected cos shape: {rope_attn.rope_cos.shape}"
print(f"RoPE cache shape: rope_cos={rope_attn.rope_cos.shape}, rope_sin={rope_attn.rope_sin.shape} ✓")

# --- 3. Full model forward pass with RoPE ---
m = ChessTransformer(
    vocab_size=1143,
    attention_factory=lambda: RoPEMultiHeadAttention(d_model=128, n_heads=4, max_seq_len=256),
    d_model=128,
    n_layers=4,
    use_sinusoidal_pe=False,   # RoPE encodes position inside attention — no separate PE needed
)

idx = torch.randint(0, 1143, (2, 64))
tgt = torch.randint(0, 1143, (2, 64))
logits, loss = m(idx, targets=tgt)

assert logits.shape == (2, 64, 1143)
loss.backward()
print(f"\nRoPE model — logits: {logits.shape} | loss: {loss.item():.4f} ✓")
print(f"Params: {m.count_parameters():,}")
print("RoPE full model ✓")

In [ ]:
import attention; importlib.reload(attention)
import model;     importlib.reload(model)

from attention import MultiHeadAttention, GroupedQueryAttention, causal_mask
from model import ChessTransformer

# --- 1. Shape + gradient check (n_heads=4, kv_heads=2 -> groups=2) ---
gqa  = GroupedQueryAttention(d_model=128, n_heads=4, kv_heads=2)
x    = torch.randn(2, 16, 128, requires_grad=True)
mask = causal_mask(16, device=torch.device('cpu'))
out  = gqa(x, mask=mask)

assert out.shape == x.shape, f"Shape mismatch: {out.shape}"
out.sum().backward()
assert x.grad is not None
print(f"GroupedQueryAttention output: {out.shape} ✓")
print("Gradients flow ✓")

# --- 2. Parameter count — GQA should have fewer params than vanilla MHA ---
vanilla_params = sum(p.numel() for p in MultiHeadAttention(128, 4).parameters())
gqa_params     = sum(p.numel() for p in gqa.parameters())
print(f"\nVanilla MHA params: {vanilla_params:,}")
print(f"GQA params (kv=2):  {gqa_params:,}  (fewer K/V params ✓)")
assert gqa_params < vanilla_params

# --- 3. Boundary cases ---
# kv_heads == n_heads -> behaves like MHA
gqa_full = GroupedQueryAttention(d_model=128, n_heads=4, kv_heads=4)
out_full = gqa_full(x.detach(), mask=mask)
assert out_full.shape == x.shape
print(f"\nGQA (kv=n_heads, equiv to MHA): {out_full.shape} ✓")

# kv_heads == 1 -> Multi-Query Attention
gqa_mqa = GroupedQueryAttention(d_model=128, n_heads=4, kv_heads=1)
out_mqa = gqa_mqa(x.detach(), mask=mask)
assert out_mqa.shape == x.shape
print(f"GQA (kv=1, equiv to MQA):      {out_mqa.shape} ✓")

# --- 4. Full model forward pass ---
m = ChessTransformer(
    vocab_size=1143,
    attention_factory=lambda: GroupedQueryAttention(d_model=128, n_heads=4, kv_heads=2),
    d_model=128,
    n_layers=4,
)

idx = torch.randint(0, 1143, (2, 64))
tgt = torch.randint(0, 1143, (2, 64))
logits, loss = m(idx, targets=tgt)

assert logits.shape == (2, 64, 1143)
loss.backward()
print(f"\nGQA model — logits: {logits.shape} | loss: {loss.item():.4f} ✓")
print(f"Params: {m.count_parameters():,}")
print("GQA full model ✓")

In [ ]:
import model; importlib.reload(model)

from model import ChessTransformer
from attention import MultiHeadAttention

m = ChessTransformer(
    vocab_size=1143,
    attention_factory=lambda: MultiHeadAttention(d_model=128, n_heads=4),
    d_model=128,
    n_layers=4,
)

idx = torch.randint(0, 1143, (2, 64))
tgt = torch.randint(0, 1143, (2, 64))

logits, loss = m(idx, targets=tgt)

print(f"Logits: {logits.shape}")   # (2, 64, 1143)
print(f"Loss:   {loss.item():.4f}")
print(f"Params: {m.count_parameters():,}")

loss.backward()
print("Gradients flow ✓")

In [ ]:
from torch.utils.data import DataLoader
from attention import MultiHeadAttention
from model import ChessTransformer

# Build model
model = ChessTransformer(
    vocab_size=tokenizer.vocab_size,
    attention_factory=lambda: MultiHeadAttention(d_model=128, n_heads=4),
    d_model=128,
    n_layers=4,
)

# Take ONE batch from the training set
x, y = next(iter(DataLoader(train_dataset, batch_size=8)))

# Optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

# Train on that one batch for 200 steps — loss should drop close to zero
model.train()
for step in range(200):
    optimizer.zero_grad()
    _, loss = model(x, targets=y)
    loss.backward()
    optimizer.step()

    if step % 20 == 0:
        print(f"step {step:3d} | loss {loss.item():.4f}")